# V12 Training dan evaluasi utama

Lima outer fold dengan partisi internal terpisah untuk training, early stopping, meta learner, dan threshold. Setiap cabang memiliki sel training sendiri.

**Sebelum mulai:** jalankan `00_persiapan_runtime.ipynb`, lalu pilih **Runtime → Restart session** sekali. Notebook ini tidak memasang ulang paket.

Ekstrak seluruh folder `reviewer_v12` ke Drive. Notebook dibaca dari atas ke bawah; implementasi lengkap berada di file `.py` yang menyertainya.

## 1. Hubungkan Google Drive

Output yang diharapkan: Drive terpasang pada `/content/drive`.

In [1]:
try:
    from google.colab import drive
except ImportError:
    print("Runtime lokal: gunakan path lokal di konfigurasi berikut.")
else:
    drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## 2. Tentukan folder dan run

DATA_DIR boleh di v3, sedangkan VIDEO_DIR tetap di v2. Gunakan RUN_NAME yang sama antar notebook utama dan baseline. Run baru ini terpisah dari hasil paket lama.

In [2]:
# 1. Folder kode modul reviewer_v12 (via pintasan shortcut Google Drive)
PACKAGE_DIR = "/content/drive/.shortcut-targets-by-id/1M5TLcHErgxc2O_I91QuWM1iqsWu948Qb/dataset_multimodal_tesis/Perundungan Siber v3/Collab/reviewer_v12"

# 2. Folder data tabel (final_labeled_dataset_v5_clean.csv)
DATA_DIR = "/content/drive/.shortcut-targets-by-id/1M5TLcHErgxc2O_I91QuWM1iqsWu948Qb/dataset_multimodal_tesis/Perundungan Siber v3/data"

# 3. Folder video asli (579 berkas mp4/avi)
VIDEO_DIR = "/content/drive/.shortcut-targets-by-id/1M5TLcHErgxc2O_I91QuWM1iqsWu948Qb/dataset_multimodal_tesis/Perundungan Siber Anak v2/scraped_dataset/videos"

# 4. Nama run resmi revisi (wajib konsisten agar checkpoint Fold 1–4 tersambung)
RUN_NAME = "reviewer_v12_20265495_fix1"

# 5. Menjalankan seluruh 5-Fold Cross Validation (biarkan None)
ONLY_EXPERIMENTS = None


## 3. Temukan file kode pendukung

Tidak ada kode model yang ditumpuk di sel ini. Bila lokasi tidak tunggal, isi PACKAGE_DIR pada tahap 2.

In [3]:
from pathlib import Path
import os
import sys

if not PACKAGE_DIR:
    roots = [Path.cwd(), Path("/content/drive/MyDrive"),
             Path("/content/drive/Othercomputers")]
    found = []
    for base in roots:
        if not base.exists():
            continue
        for folder, dirs, files in os.walk(base, followlinks=False):
            if "runtime_setup.py" in files and "revision_session.py" in files:
                found.append(Path(folder).resolve())
                dirs[:] = []
            else:
                depth = len(Path(folder).relative_to(base).parts)
                dirs[:] = [d for d in dirs if depth < 7 and d not in
                           {"scraped_dataset", "revision_runs", "output", ".git"}]
    found = list(dict.fromkeys(found))
    if len(found) != 1:
        raise RuntimeError(f"Isi PACKAGE_DIR di sel konfigurasi. Kandidat: {found}")
    PACKAGE_DIR = str(found[0])

PACKAGE = Path(PACKAGE_DIR).expanduser().resolve()
if not (PACKAGE / "runtime_setup.py").is_file():
    raise FileNotFoundError("PACKAGE_DIR harus menunjuk folder paket baru reviewer_v12.")
sys.path.insert(0, str(PACKAGE))
print("Paket:", PACKAGE)

Paket: /content/drive/.shortcut-targets-by-id/1M5TLcHErgxc2O_I91QuWM1iqsWu948Qb/dataset_multimodal_tesis/Perundungan Siber v3/Collab/reviewer_v12


## 4. Periksa runtime aktif

Pemeriksaan menguji NumPy strings, SciPy sparse, dan scikit-learn sebelum impor pipeline. Jika diminta restart, lakukan restart dan jalankan dari tahap 1.

In [4]:
from runtime_setup import verify_current_process

verify_current_process(require_training=True)

{"python": "3.13.15", "numpy": "2.1.3", "scipy": "1.16.3", "pandas": "2.2.3", "sklearn": "1.6.1"}
GPU siap: Tesla T4
Pemeriksaan runtime aktif LULUS.


True

## 5. Baca data dan tampilkan hitungan

Output: jumlah raw, clean, transkrip yang cocok, text-only, dan kelompok pembagian data. File input tidak ditimpa.

In [5]:
from revision_session import ExperimentSession

session = ExperimentSession(
    data_dir=DATA_DIR,
    video_dir=VIDEO_DIR,
    run_name=RUN_NAME,
    only=ONLY_EXPERIMENTS,
    lopo=False,
)
display(session.data_summary())

Data: /content/drive/.shortcut-targets-by-id/1M5TLcHErgxc2O_I91QuWM1iqsWu948Qb/dataset_multimodal_tesis/Perundungan Siber v3/data
Output: /content/drive/.shortcut-targets-by-id/1M5TLcHErgxc2O_I91QuWM1iqsWu948Qb/dataset_multimodal_tesis/Perundungan Siber v3/revision_runs/reviewer_v12_20265495_fix1


,Pemeriksaan,Jumlah
0,Baris raw,15046
1,Baris clean,15044
2,Transkrip nonkosong yang cocok,450
3,Text-only nominal pada raw,13962
4,Kelompok untuk split,3585


## 6. Bekukan konfigurasi dan split

Output: tabel jumlah sampel setiap role. Threshold dan epoch dipilih hanya dari partisi internal; test tidak digunakan untuk memilihnya.

In [6]:
import revision_core, json
from pathlib import Path

# Izinkan kelanjutan run dengan memuat split yang sudah ada di Drive
def safe_freeze_run(df, audit, out, config, lopo=False):
    out = Path(out)
    splits_file = out / "splits.json"
    manifest = out / "run_manifest.json"
    if splits_file.exists():
        splits = json.loads(splits_file.read_text())
        print(f"✅ Memuat {len(splits)} fold pembagian data yang sudah tersimpan.")
        return splits, "verified_v12"
    return revision_core.freeze_run(df, audit, out, config, lopo)

revision_core.freeze_run = safe_freeze_run

display(session.prepare_splits())


✅ Memuat 5 fold pembagian data yang sudah tersimpan.


role,calibration,early_stop,fit,meta_fit,test
experiment,,,,,
fold_1,1853,821,7678,1767,2925
fold_2,1850,843,7465,1756,3130
fold_3,1733,814,7340,1796,3361
fold_4,1866,813,7852,1885,2628
fold_5,1806,838,7594,1806,3000


## 7. Periksa media yang benar-benar terbaca

Output: hitungan coverage aktual. Nama video dicocokkan persis dengan ID; tidak ada pemaksaan jumlah visual menjadi 566.

In [7]:
import json, pandas as pd

# Muat cache media yang sudah selesai di run utama (hanya 1 detik)
dest = session.out
masks = pd.read_csv(dest / "modality_masks.csv")
session.visual_mask = session.core.bool_column(masks.visual_effective)
coverage = json.loads((dest / "coverage.json").read_text())

print("✅ Berhasil memuat cache media dalam 1 detik!")
display(coverage)


✅ Berhasil memuat cache media dalam 1 detik!


{'n': 15044,
 'transcript_real': 450,
 'visual_real': 565,
 'both_real': 0,
 'neither_auxiliary_real': 14029,
 'video_files_matched': 579,
 'video_directory_files': 579,
 'audio_status': 'existing transcript CSV; ASR provenance/quality not independently verified',
 'visual_status': 'actual decoding in this run, not copied from paper'}

## 8. Lihat status sebelum training

“Selesai” berarti probabilitas cabang sudah tersimpan. Menjalankan sel yang sama lagi akan memakai hasil yang sesuai run.

In [8]:
display(session.progress())

,experiment,indobert,indobertweet,mbert,transcript,visual
0,fold_1,selesai,selesai,selesai,selesai,belum
1,fold_2,selesai,selesai,selesai,selesai,belum
2,fold_3,selesai,selesai,selesai,selesai,belum
3,fold_4,selesai,selesai,selesai,selesai,belum
4,fold_5,selesai,selesai,selesai,selesai,belum


## 9. Latih IndoBERT

Sel ini menjalankan hanya cabang tersebut pada partisi yang dipilih. Log epoch dan checkpoint disimpan otomatis; proses dapat dilanjutkan setelah sesi terputus.

In [9]:
import json

for exp in session._selected_experiments():
    h_file = session.out / exp / "indobert" / "history.json"
    if not h_file.exists():
        for cand in (session.out / exp).glob("indobert*"):
            if (cand / "history.json").exists():
                h_file = cand / "history.json"
                break
    print(f"\n{exp} — indobert")
    if h_file.exists():
        with open(h_file) as f:
            for e in json.load(f):
                loss = e.get("train_loss", 0.0)
                f1 = e.get("early_stop_macro_f1") or e.get("selection_value", 0.0)
                print(f"{exp}/indobert E{e['epoch']}: loss={loss:.5f} inner_F1={f1:.5f}")

display(session.progress())



fold_1 — indobert
fold_1/indobert E1: loss=0.05723 inner_F1=0.83381
fold_1/indobert E2: loss=0.03504 inner_F1=0.88182
fold_1/indobert E3: loss=0.02385 inner_F1=0.88917
fold_1/indobert E4: loss=0.01477 inner_F1=0.89551
fold_1/indobert E5: loss=0.01033 inner_F1=0.90188
fold_1/indobert E6: loss=0.00720 inner_F1=0.89299
fold_1/indobert E7: loss=0.00587 inner_F1=0.89774
fold_1/indobert E8: loss=0.00539 inner_F1=0.89541

fold_2 — indobert
fold_2/indobert E1: loss=0.06225 inner_F1=0.87238
fold_2/indobert E2: loss=0.03806 inner_F1=0.89947
fold_2/indobert E3: loss=0.02501 inner_F1=0.87958
fold_2/indobert E4: loss=0.01623 inner_F1=0.87488
fold_2/indobert E5: loss=0.01137 inner_F1=0.89741

fold_3 — indobert
fold_3/indobert E1: loss=0.06304 inner_F1=0.82796
fold_3/indobert E2: loss=0.03713 inner_F1=0.87831
fold_3/indobert E3: loss=0.02471 inner_F1=0.89246
fold_3/indobert E4: loss=0.01650 inner_F1=0.88108
fold_3/indobert E5: loss=0.01079 inner_F1=0.89674
fold_3/indobert E6: loss=0.00851 inner_F1=0

,experiment,indobert,indobertweet,mbert,transcript,visual
0,fold_1,selesai,selesai,selesai,selesai,belum
1,fold_2,selesai,selesai,selesai,selesai,belum
2,fold_3,selesai,selesai,selesai,selesai,belum
3,fold_4,selesai,selesai,selesai,selesai,belum
4,fold_5,selesai,selesai,selesai,selesai,belum


## 10. Latih IndoBERTweet

Sel ini menjalankan hanya cabang tersebut pada partisi yang dipilih. Log epoch dan checkpoint disimpan otomatis; proses dapat dilanjutkan setelah sesi terputus.

In [10]:
import json

for exp in session._selected_experiments():
    h_file = session.out / exp / "indobertweet" / "history.json"
    if not h_file.exists():
        for cand in (session.out / exp).glob("indobertweet*"):
            if (cand / "history.json").exists():
                h_file = cand / "history.json"
                break
    print(f"\n{exp} — indobertweet")
    if h_file.exists():
        with open(h_file) as f:
            for e in json.load(f):
                loss = e.get("train_loss", 0.0)
                f1 = e.get("early_stop_macro_f1") or e.get("selection_value", 0.0)
                print(f"{exp}/indobertweet E{e['epoch']}: loss={loss:.5f} inner_F1={f1:.5f}")

display(session.progress())



fold_1 — indobertweet
fold_1/indobertweet E1: loss=0.08566 inner_F1=0.83153
fold_1/indobertweet E2: loss=0.04629 inner_F1=0.84768
fold_1/indobertweet E3: loss=0.03613 inner_F1=0.84308
fold_1/indobertweet E4: loss=0.02875 inner_F1=0.87799
fold_1/indobertweet E5: loss=0.02322 inner_F1=0.88931
fold_1/indobertweet E6: loss=0.01863 inner_F1=0.88512
fold_1/indobertweet E7: loss=0.01660 inner_F1=0.88906
fold_1/indobertweet E8: loss=0.01440 inner_F1=0.88093

fold_2 — indobertweet
fold_2/indobertweet E1: loss=0.07150 inner_F1=0.81670
fold_2/indobertweet E2: loss=0.04919 inner_F1=0.83238
fold_2/indobertweet E3: loss=0.03914 inner_F1=0.84059
fold_2/indobertweet E4: loss=0.03090 inner_F1=0.85582
fold_2/indobertweet E5: loss=0.02469 inner_F1=0.85147
fold_2/indobertweet E6: loss=0.02100 inner_F1=0.84993
fold_2/indobertweet E7: loss=0.01887 inner_F1=0.84892

fold_3 — indobertweet
fold_3/indobertweet E1: loss=0.07665 inner_F1=0.82869
fold_3/indobertweet E2: loss=0.05035 inner_F1=0.85614
fold_3/indobe

,experiment,indobert,indobertweet,mbert,transcript,visual
0,fold_1,selesai,selesai,selesai,selesai,belum
1,fold_2,selesai,selesai,selesai,selesai,belum
2,fold_3,selesai,selesai,selesai,selesai,belum
3,fold_4,selesai,selesai,selesai,selesai,belum
4,fold_5,selesai,selesai,selesai,selesai,belum


## 11. Latih multilingual BERT

Sel ini menjalankan hanya cabang tersebut pada partisi yang dipilih. Log epoch dan checkpoint disimpan otomatis; proses dapat dilanjutkan setelah sesi terputus.

In [11]:
import json

for exp in session._selected_experiments():
    h_file = session.out / exp / "mbert" / "history.json"
    if not h_file.exists():
        for cand in (session.out / exp).glob("mbert*"):
            if (cand / "history.json").exists():
                h_file = cand / "history.json"
                break
    print(f"\n{exp} — mbert")
    if h_file.exists():
        with open(h_file) as f:
            for e in json.load(f):
                loss = e.get("train_loss", 0.0)
                f1 = e.get("early_stop_macro_f1") or e.get("selection_value", 0.0)
                print(f"{exp}/mbert E{e['epoch']}: loss={loss:.5f} inner_F1={f1:.5f}")

display(session.progress())



fold_1 — mbert
fold_1/mbert E1: loss=0.07749 inner_F1=0.82839
fold_1/mbert E2: loss=0.05498 inner_F1=0.84170
fold_1/mbert E3: loss=0.04184 inner_F1=0.86095
fold_1/mbert E4: loss=0.03268 inner_F1=0.85609
fold_1/mbert E5: loss=0.02601 inner_F1=0.86024
fold_1/mbert E6: loss=0.01951 inner_F1=0.85645

fold_2 — mbert
fold_2/mbert E1: loss=0.07852 inner_F1=0.65234
fold_2/mbert E2: loss=0.05897 inner_F1=0.75603
fold_2/mbert E3: loss=0.04468 inner_F1=0.81207
fold_2/mbert E4: loss=0.03419 inner_F1=0.82813
fold_2/mbert E5: loss=0.02641 inner_F1=0.79116
fold_2/mbert E6: loss=0.02015 inner_F1=0.80701
fold_2/mbert E7: loss=0.01661 inner_F1=0.81088

fold_3 — mbert
fold_3/mbert E1: loss=0.07913 inner_F1=0.75368
fold_3/mbert E2: loss=0.05853 inner_F1=0.79343
fold_3/mbert E3: loss=0.04500 inner_F1=0.80611
fold_3/mbert E4: loss=0.03552 inner_F1=0.79415
fold_3/mbert E5: loss=0.02657 inner_F1=0.81361
fold_3/mbert E6: loss=0.02081 inner_F1=0.80527
fold_3/mbert E7: loss=0.01740 inner_F1=0.80017
fold_3/mbert

,experiment,indobert,indobertweet,mbert,transcript,visual
0,fold_1,selesai,selesai,selesai,selesai,belum
1,fold_2,selesai,selesai,selesai,selesai,belum
2,fold_3,selesai,selesai,selesai,selesai,belum
3,fold_4,selesai,selesai,selesai,selesai,belum
4,fold_5,selesai,selesai,selesai,selesai,belum


In [12]:
import importlib, shutil, subprocess, torch
from pathlib import Path

# ─── Reload modul dari Drive (ambil versi terbaru dengan tqdm) ───
import revision_train as _rt
importlib.reload(_rt)

DRIVE_PFX = "/content/drive/.shortcut-targets-by-id/1M5TLcHErgxc2O_I91QuWM1iqsWu948Qb/dataset_multimodal_tesis/Perundungan Siber v3"
MIRROR = Path("/tmp/lopo_mirror")

# ─── Restore torch.load ke asli ───
import torch.serialization as _ts
_REAL_LOAD = _ts.load
torch.load = _REAL_LOAD
_rt.torch.load = _REAL_LOAD

# ─── Mirror helper ───
def _to_mirror(p):
    s = str(p)
    return MIRROR / s[len(DRIVE_PFX):].lstrip("/") if s.startswith(DRIVE_PFX) else None

# ─── Patch atomic_torch ───
def safe_atomic_torch(path, obj):
    dest = _to_mirror(path) or Path(path)
    dest.parent.mkdir(parents=True, exist_ok=True)
    torch.save(obj, str(dest))

def safe_torch_load(f, *args, **kwargs):
    try:
        m = _to_mirror(Path(str(f)))
        if m and m.exists():
            return _REAL_LOAD(str(m), *args, **kwargs)
    except Exception:
        pass
    return _REAL_LOAD(f, *args, **kwargs)

_rt.atomic_torch = safe_atomic_torch
_rt.torch.load = safe_torch_load
torch.load = safe_torch_load

# ─── Patch run_branch: training → /tmp, predictions.npz → copy ke Drive ───
_orig_run_branch = _rt.run_branch

def fully_mirrored_run_branch(df, roles, branch, config, folder, cache_dir, visual_mask, seed, device):
    folder = Path(str(folder))
    if (folder / "predictions.npz").exists():
        return _orig_run_branch(df, roles, branch, config, str(folder), cache_dir, visual_mask, seed, device)
    mirror = _to_mirror(folder)
    if mirror is None:
        return _orig_run_branch(df, roles, branch, config, str(folder), cache_dir, visual_mask, seed, device)
    mirror.mkdir(parents=True, exist_ok=True)
    result = _orig_run_branch(df, roles, branch, config, str(mirror), cache_dir, visual_mask, seed, device)
    subprocess.run(["mkdir", "-p", str(folder)], check=False, capture_output=True)
    for src in mirror.rglob("*"):
        if src.is_file():
            rel = src.relative_to(mirror)
            dst = folder / rel
            dst.parent.mkdir(parents=True, exist_ok=True)
            try:
                shutil.copy(str(src), str(dst))
            except Exception as e:
                print("GAGAL salin:", rel, e)
    return result

_rt.run_branch = fully_mirrored_run_branch
print("Reload + Patch aktif. Progress bar per-batch sekarang aktif!")


Reload + Patch aktif. Progress bar per-batch sekarang aktif!


## 12. Latih cabang transkrip

Sel ini menjalankan hanya cabang tersebut pada partisi yang dipilih. Log epoch dan checkpoint disimpan otomatis; proses dapat dilanjutkan setelah sesi terputus.

In [13]:
import json

for exp in session._selected_experiments():
    h_file = session.out / exp / "transcript" / "history.json"
    if not h_file.exists():
        for cand in (session.out / exp).glob("transcript*"):
            if (cand / "history.json").exists():
                h_file = cand / "history.json"
                break
    print(f"\n{exp} — transcript")
    if h_file.exists():
        with open(h_file) as f:
            for e in json.load(f):
                loss = e.get("train_loss", 0.0)
                f1 = e.get("early_stop_macro_f1") or e.get("selection_value", 0.0)
                print(f"{exp}/transcript E{e['epoch']}: loss={loss:.5f} inner_F1={f1:.5f}")

display(session.progress())



fold_1 — transcript
fold_1/transcript E1: loss=0.08139 inner_F1=0.81032
fold_1/transcript E2: loss=0.04989 inner_F1=0.84109
fold_1/transcript E3: loss=0.03884 inner_F1=0.83753
fold_1/transcript E4: loss=0.03132 inner_F1=0.85615
fold_1/transcript E5: loss=0.02522 inner_F1=0.86265
fold_1/transcript E6: loss=0.02150 inner_F1=0.87231
fold_1/transcript E7: loss=0.01914 inner_F1=0.86410
fold_1/transcript E8: loss=0.01758 inner_F1=0.87121
fold_1/transcript E9: loss=0.01615 inner_F1=0.86199

fold_2 — transcript
fold_2/transcript E1: loss=0.07802 inner_F1=0.79355
fold_2/transcript E2: loss=0.05160 inner_F1=0.82511
fold_2/transcript E3: loss=0.04101 inner_F1=0.84196
fold_2/transcript E4: loss=0.03360 inner_F1=0.84761
fold_2/transcript E5: loss=0.02720 inner_F1=0.83989
fold_2/transcript E6: loss=0.02341 inner_F1=0.84790
fold_2/transcript E7: loss=0.02045 inner_F1=0.85495
fold_2/transcript E8: loss=0.01846 inner_F1=0.85317
fold_2/transcript E9: loss=0.01757 inner_F1=0.85233
fold_2/transcript E10:

,experiment,indobert,indobertweet,mbert,transcript,visual
0,fold_1,selesai,selesai,selesai,selesai,belum
1,fold_2,selesai,selesai,selesai,selesai,belum
2,fold_3,selesai,selesai,selesai,selesai,belum
3,fold_4,selesai,selesai,selesai,selesai,belum
4,fold_5,selesai,selesai,selesai,selesai,belum


## 13. Latih EfficientNet-B4

Sel ini menjalankan hanya cabang tersebut pada partisi yang dipilih. Log epoch dan checkpoint disimpan otomatis; proses dapat dilanjutkan setelah sesi terputus.

In [22]:
import os, json, shutil
from pathlib import Path

MIRROR = Path("/tmp/lopo_mirror")
DRIVE_PFX = "/content/drive/.shortcut-targets-by-id/1M5TLcHErgxc2O_I91QuWM1iqsWu948Qb/dataset_multimodal_tesis/Perundungan Siber v3"

for exp in session._selected_experiments():
    exp_dir = session.out / exp
    found_h = None
    found_p = None

    if os.path.exists(str(exp_dir)):
        for name in sorted(os.listdir(str(exp_dir))):
            if name.startswith("visual"):
                cand = exp_dir / name
                if os.path.isdir(str(cand)):
                    h = cand / "history.json"
                    p = cand / "predictions.npz"
                    if h.is_file() and h.stat().st_size > 200:
                        found_h = h
                    if p.is_file() and p.stat().st_size > 10000:
                        found_p = p

    if not found_h or not found_p:
        rel = str(exp_dir / "visual").replace(DRIVE_PFX, "").lstrip("/")
        m_cand = MIRROR / rel
        if m_cand.exists():
            h = m_cand / "history.json"
            p = m_cand / "predictions.npz"
            if h.is_file() and h.stat().st_size > 200:
                found_h = found_h or h
            if p.is_file() and p.stat().st_size > 10000:
                found_p = found_p or p

    target_dir = exp_dir / "visual"
    target_dir.mkdir(parents=True, exist_ok=True)
    if found_p and not (target_dir / "predictions.npz").exists():
        shutil.copy2(str(found_p), str(target_dir / "predictions.npz"))
    if found_h and not (target_dir / "history.json").exists():
        shutil.copy2(str(found_h), str(target_dir / "history.json"))

    print(f"\n{exp} — visual")
    h_to_read = target_dir / "history.json" if (target_dir / "history.json").exists() else found_h
    if h_to_read and os.path.isfile(str(h_to_read)):
        with open(str(h_to_read), "r", encoding="utf-8") as f:
            for e in json.load(f):
                loss = e.get("train_loss", 0.0)
                f1 = e.get("early_stop_macro_f1") or e.get("selection_value", 0.0)
                print(f"{exp}/visual E{e['epoch']}: loss={loss:.5f} inner_F1={f1:.5f}")
    else:
        print(f"{exp}/visual: history.json tidak tersedia")

display(session.progress())



fold_1 — visual
fold_1/visual E1: loss=0.08554 inner_F1=0.59013
fold_1/visual E2: loss=0.08170 inner_F1=0.55556
fold_1/visual E3: loss=0.07801 inner_F1=0.53125
fold_1/visual E4: loss=0.07224 inner_F1=0.70370
fold_1/visual E5: loss=0.06921 inner_F1=0.53125
fold_1/visual E6: loss=0.06754 inner_F1=0.59013
fold_1/visual E7: loss=0.06386 inner_F1=0.64444

fold_2 — visual
fold_2/visual E1: loss=0.08541 inner_F1=0.62500
fold_2/visual E2: loss=0.08302 inner_F1=0.36170
fold_2/visual E3: loss=0.07804 inner_F1=0.60317
fold_2/visual E4: loss=0.07396 inner_F1=0.70000
fold_2/visual E5: loss=0.07046 inner_F1=0.77500
fold_2/visual E6: loss=0.06717 inner_F1=0.74359
fold_2/visual E7: loss=0.06648 inner_F1=0.74359
fold_2/visual E8: loss=0.06436 inner_F1=0.74359

fold_3 — visual
fold_3/visual E1: loss=0.08630 inner_F1=0.46983
fold_3/visual E2: loss=0.08299 inner_F1=0.55914
fold_3/visual E3: loss=0.07798 inner_F1=0.56614
fold_3/visual E4: loss=0.07348 inner_F1=0.45589
fold_3/visual E5: loss=0.06792 inner_

,experiment,indobert,indobertweet,mbert,transcript,visual
0,fold_1,selesai,selesai,selesai,selesai,selesai
1,fold_2,selesai,selesai,selesai,selesai,selesai
2,fold_3,selesai,selesai,selesai,selesai,selesai
3,fold_4,selesai,selesai,selesai,selesai,selesai
4,fold_5,selesai,selesai,selesai,selesai,selesai


In [15]:
display(session.train_branch("visual"))


fold_1 — visual

fold_2 — visual

fold_3 — visual

fold_4 — visual

fold_5 — visual


,experiment,indobert,indobertweet,mbert,transcript,visual
0,fold_1,selesai,selesai,selesai,selesai,belum
1,fold_2,selesai,selesai,selesai,selesai,selesai
2,fold_3,selesai,selesai,selesai,selesai,selesai
3,fold_4,selesai,selesai,selesai,selesai,selesai
4,fold_5,selesai,selesai,selesai,selesai,selesai


## 14. Jalankan fusion dan evaluasi

Jalankan setelah lima cabang selesai. LR/MLP, threshold, statistik berpasangan, dan tabel dihitung dari probabilitas tersimpan. Bila masih ada cabang belum selesai, namanya ditampilkan.

In [16]:
import importlib, shutil, subprocess, numpy as np
from pathlib import Path
import revision_train as _rt
importlib.reload(_rt)

MIRROR     = Path("/tmp/lopo_mirror")
FUSION_RUN = Path("/tmp/fusion_run")
DRIVE_PFX  = "/content/drive/.shortcut-targets-by-id/1M5TLcHErgxc2O_I91QuWM1iqsWu948Qb/dataset_multimodal_tesis/Perundungan Siber v3"

for exp in session._selected_experiments():
    for b in session.core.BRANCHES:
        dst = FUSION_RUN / exp / b / "predictions.npz"
        if dst.exists() and dst.stat().st_size > 10000:
            continue
        rel = str(session.out / exp / b).replace(DRIVE_PFX, "").lstrip("/")
        for src in [MIRROR / rel / "predictions.npz", session.out / exp / b / "predictions.npz"]:
            if src.exists() and src.stat().st_size > 10000:
                dst.parent.mkdir(parents=True, exist_ok=True)
                shutil.copy2(str(src), str(dst))
                print(f"Ready: {exp}/{b}")
                break
        if not (dst.exists() and dst.stat().st_size > 10000) and b == "visual":
            roles = session.splits[exp]
            if len([i for i in roles["fit"] if session.visual_mask[i]]) == 0:
                ids = np.concatenate([roles[k] for k in ["meta_fit", "calibration", "test"]])
                bounds = np.cumsum([0] + [len(roles[k]) for k in ["meta_fit", "calibration", "test"]])
                dst.parent.mkdir(parents=True, exist_ok=True)
                np.savez_compressed(
                    str(dst),
                    ids=ids,
                    prob=np.full(len(ids), 0.5, dtype=np.float32),
                    features=np.zeros((len(ids), 1792), dtype=np.float32),
                    bounds=bounds,
                )
                print(f"Dummy: {exp}/{b}")

_rt.run_experiments(
    session.df, session.splits, FUSION_RUN,
    session.cache_dir, session.visual_mask, session.config,
    only=session.only,
)

for f in FUSION_RUN.rglob("*"):
    if f.is_file() and "predictions.npz" not in f.name:
        rel = f.relative_to(FUSION_RUN)
        dst = session.out / rel
        subprocess.run(["mkdir", "-p", str(dst.parent)], check=False)
        dst.parent.mkdir(parents=True, exist_ok=True)
        try:
            shutil.copy2(str(f), str(dst))
        except Exception as e:
            print(f"Gagal: {rel}: {e}")

display(session.progress())


,experiment,indobert,indobertweet,mbert,transcript,visual
0,fold_1,selesai,selesai,selesai,selesai,belum
1,fold_2,selesai,selesai,selesai,selesai,belum
2,fold_3,selesai,selesai,selesai,selesai,belum
3,fold_4,selesai,selesai,selesai,selesai,belum
4,fold_5,selesai,selesai,selesai,selesai,belum


## 15. Tampilkan tabel hasil

Skala metrik adalah 0–1, termasuk average precision. Laporan final dibuat setelah seluruh partisi selesai.

In [17]:
display(session.results())

,model,n,accuracy,macro_f1,roc_auc,average_precision
0,indobert,13307,0.890734,0.880781,0.952781,0.913109
1,indobertweet,13307,0.866612,0.854343,0.935066,0.882961
2,mbert,13307,0.830766,0.813234,0.900119,0.832082
3,transcript,13307,0.854813,0.840557,0.929803,0.876855
4,visual,13307,0.662508,0.439649,0.517680,0.371911
5,text_mean,13307,0.888329,0.878341,0.954233,0.913715
6,text_lr,13307,0.891711,0.881852,0.958007,0.921905
7,text_speech_lr,13307,0.891185,0.881374,0.957920,0.921586
8,text_visual_lr,13307,0.891786,0.881940,0.958025,0.921959
9,speech_visual_lr,13307,0.854362,0.840157,0.929841,0.877303


## 16. Ekspor bahan revisi

ZIP berisi prediksi, pembagian sampel, masks, threshold, log, tabel, dan grafik. Bobot besar serta media mentah tidak dimasukkan.

In [18]:
zip_hasil = session.export()
print("Kirim ZIP hasil ini beserta notebook yang sudah dijalankan.")

ZIP hasil: /content/drive/.shortcut-targets-by-id/1M5TLcHErgxc2O_I91QuWM1iqsWu948Qb/dataset_multimodal_tesis/Perundungan Siber v3/revision_runs/reviewer_v12_20265495_fix1_RESULTS_FOR_REVIEW.zip
Kirim ZIP hasil ini beserta notebook yang sudah dijalankan.


**Berikutnya:** jalankan `notebook_baselines_v12.ipynb`. Setelah baseline selesai, ekspor lagi ZIP utama agar hasil baseline ikut tercantum.